<a href="https://colab.research.google.com/github/vidaechea/lidr-Master-ai-engineering/blob/development/session01/session_01_2_api_anthropic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from anthropic import (
    Anthropic,
    AuthenticationError,
    RateLimitError,
    APIConnectionError,
    BadRequestError,
    InternalServerError,
)
from google.colab import userdata

# Retrieve API key from Colab secrets
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
client = Anthropic(api_key=ANTHROPIC_API_KEY)

# claude-haiku-4-5 pricing (check periodically)
PRICING = {
    "claude-haiku-4-5-20251001": {"input": 1.00, "output": 5.00},
}

def query_claude(message, system_prompt=None, model="claude-haiku-4-5-20251001", temperature=0.3):
    """
    Make a call to the Anthropic Messages API and return
    the response along with relevant metadata.
    """
    try:
        kwargs = {
            "model": model,
            "messages": [{"role": "user", "content": message}],
            "max_tokens": 1000,
            "temperature": temperature,
        }
        if system_prompt:
            kwargs["system"] = system_prompt

        response = client.messages.create(**kwargs)

        # Check if the response was truncated
        if response.stop_reason == "max_tokens":
            print("⚠️  Response truncated")

        # Calculate cost
        prices = PRICING.get(model, {"input": 0, "output": 0})
        cost = (
            (response.usage.input_tokens / 1_000_000) * prices["input"] +
            (response.usage.output_tokens / 1_000_000) * prices["output"]
        )

        return {
            "content": response.content[0].text,
            "model": response.model,
            "id": response.id,
            "input_tokens": response.usage.input_tokens,
            "output_tokens": response.usage.output_tokens,
            "stop_reason": response.stop_reason,
            "cost_usd": cost,
        }

    except AuthenticationError:
        return {"error": "Invalid or missing API key"}
    except RateLimitError:
        return {"error": "Rate limit reached or insufficient credit"}
    except BadRequestError as e:
        return {"error": f"Invalid request: {e.message}"}
    except (APIConnectionError, InternalServerError):
        return {"error": "Connection or server error"}

# Usage
result = query_claude(
    message="How long would a PostgreSQL to Aurora migration take?",
    system_prompt="You are a software estimation consultant. Be concise."
)

if "error" in result:
    print(f"Error: {result['error']}")
else:
    print(result["content"])
    print(f"\n--- Metadata ---")
    print(f"Model: {result['model']}")
    print(f"ID: {result['id']}")
    print(f"Tokens: {result['input_tokens']} input + {result['output_tokens']} output")
    print(f"Stop reason: {result['stop_reason']}")
    print(f"Cost: ${result['cost_usd']:.6f}")

# PostgreSQL to Aurora Migration Timeline

**Typical range: 2-8 weeks** depending on:

## Key Variables

| Factor | Impact |
|--------|--------|
| **Database size** | <100GB: 1-2 weeks; >1TB: 4-8 weeks |
| **Complexity** | Simple schema: faster; Custom functions/extensions: slower |
| **Downtime tolerance** | Zero-downtime: adds 1-2 weeks; Acceptable downtime: faster |
| **Testing requirements** | Basic: included; Comprehensive: +1-2 weeks |

## Typical Phases

1. **Assessment & Planning** (3-5 days)
2. **Schema conversion** (3-7 days)
3. **Data migration** (2-14 days depending on size)
4. **Testing & validation** (5-10 days)
5. **Cutover & monitoring** (2-5 days)

## Accelerators
- Use AWS DMS (Database Migration Service) - reduces manual work
- Minimal schema changes needed
- Good test environment parity

## Risk Factors
- Unsupported PostgreSQL extensions
- Complex stored procedures
- Large data volumes with tight cutover windows
- Compliance/validation requirements

**Recommendatio